# 00 | Smoke Test:環境與資料庫連線驗證
目的：驗證 Python 環境可透過 sqlalchemy 讀取 PostgreSQL raw 層，並取得基準流失率。<br/>
cell 1-2：煙霧測試 (驗證環境通道 + 拿到基準流失率數字)<br/>
cell 3–5： [M0-7] 的兩種資料載入方式(「`\copy` + 手寫 DDL」 vs 「pandas `to_sql`寫入」 )對照實驗<br/>

SQLAlchemy 補充說明：<br/>
- 1.Python 透過 SQLAlchemy (此處僅用其引擎/連線層，未用 ORM 功能)這個工具來連線並操作 PostgreSQL；pandas 的 read_sql 以 SQLAlchemy engine 為官方標準 interface (pandas 官方文件指定：「pd.read_sql(sql, con) 」的第二個參數 con(連線)，要傳入 SQLAlchemy 的 engine(或 connection))。
- 2.本案只有用 SQLAlchemy 的 Core/引擎層：管理連線池、將 SQL 指令送到資料庫執行、當 pandas 和 postgresql 的對接 interface。
- 3.ORM = 把資料表映射成 Python 類別、用物件操作代替寫 SQL。本案沒有使用 ORM 功能，而是自己寫原始 SQL 指令來與資料庫互動。

In [1]:
# cell 1：連線驗證 + 分組計數
import sys
import pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # 把專案根目錄加進模組搜尋路徑，才 import 得到 src/

# 第一道測試：執行 db.py (laod_dotenv 讀 .env、failfast檢查)：若.env沒建好，執行這行就會發現
from src.db import get_engine
import pandas as pd

# 拿到 singleton Engine，但此時還沒真的連資料庫(lazy singleton)
engine = get_engine()

# 第一次進資料庫：驗證「資料庫連線正常」、「raw層的 schema、表、資料都在」
# 1.分組：GROUP BY attrition_flag : 按流失標籤分組。這欄只有兩種值('Existing Customer' / 'Attrited Customer')，所以依這兩個種類分出兩組。
# 2.分別計算「個別組別」內「有幾筆資料」
# 預期輸出兩列：Attrited 1627 / Existing 8500 (加總 10,127)。這裡其實隱含一次資料完整性核對 (肉眼相加 1627 和 8500 就能看出總共有 10127 筆)
df = pd.read_sql(
    "SELECT attrition_flag, COUNT(*) AS n FROM raw.bank_churners GROUP BY attrition_flag",
    engine,
)
df

,attrition_flag,n
0,Attrited Customer,1627
1,Existing Customer,8500


In [2]:
# cell 2：算基準流失率：三次交叉驗證的第一次：路徑 = DB 分組聚合 (cell 1) + pandas 相除 (cell 2) (混合路徑)
# 本段語法目標：對 cell 1 得出的 DataFrame ( 兩列 、兩欄 ) 運算 (不是對一萬比原始資料運算，因為已在 cell 1 的 DB 端做完聚合，Python只拿到 df(兩個數字))。df 是：
#        attrition_flag        n
# 0    Attrited Customer     1627
# 1    Existing Customer     8500

# 語法拆解：
# (1)比較整欄 (向量化比較) ( df["attrition_flag"] == "Attrited Customer" )：做「向量化比較」，一次比較 attrition_flag 這個欄位的整欄，得出一串布林值 series ( Pandas在底層幫你逐筆比較運算，不用自己寫迴圈來運算 )。本例會得出：
# 0    True
# 1    False

# df.loc[mask, "n"] 是一步完成「篩列 + 選欄」，所以(2)和(3)實際是同一個動作完成，此處拆成兩步只是輔助理解概念)
# (2)用布林遮罩篩選列 (df.loc[mask])：保留 True 的列，再取 n 這欄 ( 選出 Attrited Customer 和那列的 n (1627) )。本例會得出：
# attrition_flag        n
# Attrited Customer     1627
# (3) 取 n 這個欄位 (df.loc[mask, "n"])：從 (2) 的結果中取出 n 欄位。本例會得出：
# 0    1627
# Name: n, dtype: int64
# 說明：df.loc[mask, "n"] 的產出是一個 Series(單欄資料)， Series 印出時自帶兩個附屬資訊 Name 和 dtype。左邊的 0 是這個 df 中原本屬於 1627 的原列索引。Name 屬性指：Series 當時是從 n 這個欄位取出來的，印出來顯示 Name: n (標記身份，不影響運算)。

# (4) 將 (3) 的結果加總 (.sum())：本例只有一列，所以本例會得出：
# 1627
# (5) 最後 1627 除以 df["n"].sum()，本例的 n 加總 (包括 Existing Customer 和 Attrited Customer) 是 10127。1627 / 10127，本例會得出：0.1607。
# (6) 0.1607 用 :.2% (乘 100、留2位小數、加%符號)得出最終結果 16.07%。
# (7) 語法補充：.loc[列條件, 欄名稱] 就是：「先決定要哪些列，再決定要哪些欄」。

churn_rate = df.loc[df["attrition_flag"] == "Attrited Customer", "n"].sum() / df["n"].sum()
print(f"基準流失率:{churn_rate:.2%}")

基準流失率:16.07%


In [3]:
# cell 3：to_sql 對照實驗：載入資料

# 用 pandas 實際走一次載入流程：到 data 資料夾讀這個 csv 檔進記憶體，並在這裡用 read_csv 寫入資料到暫存表，之後用這張暫存資料表(pandas to_sql) 與 \copy 版資料表(手寫ddl)比對 (比對兩種資料載入方式的差異)
# 這條路是從源頭 CSV 另外走一條載入路線，這就是兩條不同路線(pandas to_sql vs 手寫ddl) 對照的實驗意義：因為 read_csv 會自動推斷型別，而手寫 DDL 建表則是預設全 TEXT 型別，由此比較對照不同的行為。
raw_df = pd.read_csv("../data/raw/BankChurners.csv")

# 把末兩欄原始欄名改成短名，因為末兩欄原始欄名超過 PostgreSQL 63 字元上限，截斷後兩個欄位名前 63 字元相同會報錯 (重複欄名)
raw_df.columns.values[-2] = "nb_classifier_prob_1"
raw_df.columns.values[-1] = "nb_classifier_prob_2"

# 將 raw_df 這個 DataFrame 寫入資料庫成為一張暫存表：pandas 依推斷的型別自動生成 CREATE TABLE 再逐批 INSERT (if_exists="replace" 保證可重跑；index=False 不把列索引寫成多餘欄位)
raw_df.to_sql("bank_churners_pandas", engine, schema="raw", if_exists="replace", index=False)

# 筆數驗證：10127 (任何載入動作做完後，都做一次資料筆數驗證)
pd.read_sql("SELECT COUNT(*) FROM raw.bank_churners_pandas", engine)
# 預期 10127

,count
0,10127


In [4]:
# cell 4：型別比對
# 查資料庫的 metadata，去 raw 這個 schema 裡查「所有欄位名是 customer_age (大小寫不分) 的欄位登記的型別」。
# 輸出兩列並排：\copy 版是 text，to_sql 版是 bigint
# 比對型別
pd.read_sql("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'raw' AND LOWER(column_name) = 'customer_age'
""", engine)

,table_name,column_name,data_type
0,bank_churners_pandas,Customer_Age,bigint
1,bank_churners,customer_age,text


In [5]:
# cell 5：清掉暫存表

# engine.begin() 開 transactoin: 正常走完 transaction 就自動 COMMIT、 transaction 出錯就自動 ROLLBACK。text() 是 SQLAlchemy 執行原始 SQL 字串的必要包裝
# 比對實驗做完就清掉暫存表，raw 層保持乾淨

from sqlalchemy import text
with engine.begin() as conn:
    conn.execute(text("DROP TABLE raw.bank_churners_pandas"))